# 06 — Scenario analysis (Phase 6)

Phase 5 said who is underserved and where. This notebook asks the
planning question: **if an announced project existed, who could reach
how many more jobs?** Each scenario is a GeoJSON file in
`config/scenarios/` — a drawn alignment plus a handful of assumed
operating parameters. The pipeline merges the drawing into the baseline
network, rebuilds ONLY the affected mode's travel times with the Phase 4
code, and stores per-hex `baseline / scenario / delta` columns. Because
a scenario only adds arcs (or speeds edges up), **every delta is ≥ 0 by
construction** — and with nothing drawn, the machinery reproduces the
stored Phase 4 matrix bit for bit (both facts are pinned by tests).

Read the deltas with the caveats the report carries:

- **Potential access, not ridership.** No demand model, no mode choice:
  the claim is about jobs a resident *could* reach within a time budget.
- **The transit baseline is a lower bound** (D-044): where it misses
  real routes, part of a "gain" already exists on the street today.
- **Alignments are illustrative** — anchored to announced projects and
  drawn from OSM street geometry, with sources inside each file; station
  names beyond Kalkaman are placeholders.
- **Jobs stay fixed** at the Phase 3 surface: no station-area
  development, no relocation.

By default this runs on the committed sample (a 2 km window, one demo
scenario); set `ALMATY_CONFIG=config/config.yaml` for the full city and
the three announced projects.


In [ ]:
import json
import os
import re
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import patheffects
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_hex, to_rgb

from almaty_access.access.build import build_measures, build_times
from almaty_access.config import load_config
from almaty_access.equity.build import SUMMARY_FILE as EQUITY_SUMMARY_FILE
from almaty_access.equity.build import build_equity
from almaty_access.equity.inequality import gini, lorenz_curve
from almaty_access.prepare.transit import prepare_transit
from almaty_access.scenarios.build import (
    GAINS_FILE,
    REPORT_FILE,
    SCENARIO_FILE_TEMPLATE,
    SUMMARY_FILE,
    affected_columns,
    build_scenarios,
    headline_measure,
)
from almaty_access.scenarios.definition import load_scenarios

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
CONFIG_PATH = Path(os.environ.get("ALMATY_CONFIG", REPO / "config" / "config.sample.yaml"))
config = load_config(CONFIG_PATH)
prepare_transit(config)  # all cached
build_times(config)
build_measures(config)
build_equity(config)
build_scenarios(config)

P = config.paths.processed
A = config.h3.analysis_resolution
RASTER, CALIBRATED = "raster", "calibrated"
report = json.loads((P / REPORT_FILE).read_text("utf-8"))
HAS_CALIBRATED = "method" in report["weightings"].get(CALIBRATED, {})
WEIGHTINGS = [RASTER, CALIBRATED] if HAS_CALIBRATED else [RASTER]
summary = pd.read_parquet(P / SUMMARY_FILE)
gains = pd.read_parquet(P / GAINS_FILE)
scenarios = {s.name: s for s in load_scenarios(config)}
hex_tables = {
    name: pd.read_parquet(P / SCENARIO_FILE_TEMPLATE.format(name=name, res=A))
    for name in report["scenarios"]
}
grid = gpd.read_parquet(P / f"grid_r{A}.parquet")[["h3", "geometry"]]
district_shapes = gpd.read_parquet(config.paths.interim / "boundary_districts.parquet")
access = pd.read_parquet(P / f"access_r{A}.parquet")

#: scenario -> (headline measure, threshold, primary mode)
HEADLINE = {
    name: (*headline_measure(config, entry["modes"][0]), entry["modes"][0])
    for name, entry in report["scenarios"].items()
}
ORDER = sorted(report["scenarios"])  # fixed order everywhere, incl. colors

print(f"config: {CONFIG_PATH.name} — {len(ORDER)} scenario(s)")
print(f"total jobs at r{A}: {report['total_jobs']:,.0f}")
print()
for name in ORDER:
    print(f"{name}:")
    print(" ", report["scenarios"][name]["headline"])

## One small chart system

The same system as notebook 05, so the whole project reads as a family:
**one hue (blue) carries magnitude**, the **blue ↔ red pair carries
polarity** (inequality falling vs rising), distinct hues appear only to
tell scenarios apart — in one fixed order with direct labels — and text
stays in ink. The table behind every chart is printed with it, so no
number has to be read from color alone.


In [ ]:
SURFACE, INK, INK_2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, NEUTRAL = "#e1e0d9", "#c3c2b7", "#f0efec"
BLUE, ORANGE, AQUA, YELLOW, RED = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e34948"
LIGHT_BLUE = "#86b6ef"
SEQ_BLUE = LinearSegmentedColormap.from_list(
    "seq_blue", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
)
# Fixed scenario order (sorted names) -> fixed hues, validated in NB05.
SCENARIO_COLOR = {name: color for name, color in zip(ORDER, [BLUE, ORANGE, AQUA], strict=False)}

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "font.family": "sans-serif",
        "font.size": 9.5,
        "text.color": INK,
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.titlesize": 10.5,
        "axes.titleweight": "semibold",
        "axes.titlelocation": "left",
        "axes.edgecolor": AXIS,
        "axes.linewidth": 1.0,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "xtick.labelcolor": INK_2,
        "ytick.labelcolor": INK_2,
        "grid.color": GRID,
        "grid.linewidth": 1.0,
        "grid.linestyle": "-",
        "legend.frameon": False,
        "legend.fontsize": 9,
    }
)


def blend(color: str, alpha: float) -> str:
    """The colour at the given opacity over the chart surface, as a hex."""
    top, base = np.array(to_rgb(color)), np.array(to_rgb(SURFACE))
    return to_hex(alpha * top + (1 - alpha) * base)


def style(ax, grid: str | None = "y") -> None:
    """Recessive chrome: no box, hairline solid grid behind the marks."""
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(length=0)
    ax.set_axisbelow(True)
    if grid:
        ax.grid(axis=grid)


def measure_label(measure: str) -> str:
    job = re.fullmatch(r"acc(?:pct)?_jobs_(\w+?)_(\d+)(?:_delta)?", measure)
    if job:
        return f"{job.group(1).capitalize()} · {job.group(2)} min"
    return measure


def short_title(name: str) -> str:
    """Titles short enough for subplot headers and tick labels."""
    return report["scenarios"][name]["title"].split(":")[0].split(" (")[0].strip()


def base_map(ax, frame: gpd.GeoDataFrame, title: str) -> None:
    """District outlines and names for orientation; no axes."""
    district_shapes.boundary.plot(ax=ax, color=AXIS, linewidth=0.8, zorder=3)
    bounds = frame.total_bounds
    for row in district_shapes.itertuples():
        point = row.geometry.representative_point()
        if bounds[0] <= point.x <= bounds[2] and bounds[1] <= point.y <= bounds[3]:
            label = ax.text(
                point.x,
                point.y,
                str(row.district).capitalize(),
                fontsize=8,
                color=INK,
                ha="center",
                va="center",
                zorder=4,
            )
            label.set_path_effects([patheffects.withStroke(linewidth=2.5, foreground=SURFACE)])
    pad_x, pad_y = (bounds[2] - bounds[0]) * 0.03, (bounds[3] - bounds[1]) * 0.03
    ax.set_xlim(bounds[0] - pad_x, bounds[2] + pad_x)
    ax.set_ylim(bounds[1] - pad_y, bounds[3] + pad_y)
    ax.set_title(title)
    ax.set_axis_off()


def to_storage(geometry_m):
    """Metric scenario geometry -> storage CRS for mapping."""
    return gpd.GeoSeries([geometry_m], crs=config.crs.metric).to_crs(config.crs.storage).iloc[0]

## What was drawn

Each scenario is literally the geometry below plus a few properties
(mode, assumed speed, headway, corridor width). The metro extension
*extends* the existing line 1 relations — riders stay on the same train,
so no phantom transfer wait is charged at the old terminus; the BRT is a
new bus route with synthesized stops every 800 m; the bike corridors
raise the flat riding speed on matched street edges. Redraw a file, run
`almaty-access scenarios`, and every figure below updates.


In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 7.2))
for name in ORDER:
    scenario = scenarios[name]
    color = SCENARIO_COLOR[name]
    for line in scenario.lines:
        geom = to_storage(line.geometry_m)
        ax.plot(*geom.xy, color=color, linewidth=2.6, solid_capstyle="round", zorder=5)
        for index, station in enumerate(line.stations):
            ax.scatter(
                station.lon,
                station.lat,
                s=14 if station.synthesized else 34,
                color=color,
                edgecolor=SURFACE,
                linewidth=1.4,
                zorder=6,
            )
            if not station.synthesized:
                # Alternate label sides; the file's "(placeholder)" note
                # stays in the data, the map keeps the name readable.
                label = ax.annotate(
                    station.name.split(" (")[0],
                    (station.lon, station.lat),
                    textcoords="offset points",
                    xytext=(6, 6 if index % 2 == 0 else -13),
                    fontsize=8,
                    color=INK,
                )
                label.set_path_effects([patheffects.withStroke(linewidth=2.5, foreground=SURFACE)])
    for boost in scenario.boosts:
        geom = to_storage(boost.geometry_m)
        ax.plot(*geom.xy, color=color, linewidth=2.2, solid_capstyle="round", zorder=5)

# Context: the baseline transit stops, faint.
stops = gpd.read_parquet(config.paths.interim / "transit_stops.parquet")
ax.scatter(stops.geometry.x, stops.geometry.y, s=3, color=MUTED, alpha=0.45, zorder=2)

inhabited_frame = grid.merge(access[["h3", "in_city"]], on="h3")
base_map(ax, inhabited_frame[inhabited_frame["in_city"]], "The drawn scenarios")
handles = [
    plt.Line2D([], [], color=SCENARIO_COLOR[name], linewidth=2.6, label=short_title(name))
    for name in ORDER
]
handles.append(
    plt.Line2D(
        [],
        [],
        color=MUTED,
        marker="o",
        markersize=3,
        linewidth=0,
        label="baseline transit stops (OSM)",
    )
)
ax.legend(handles=handles, loc="upper left")
plt.show()

for name in ORDER:
    entry = report["scenarios"][name]
    network = entry["network"]
    parts = []
    if "transit" in network:
        n = network["transit"]
        parts.append(
            f"+{n['n_new_stops']} stops, +{n['n_new_routes']} routes, +{n['n_new_legs']} legs"
        )
    if "bike" in network:
        n = network["bike"]
        parts.append(
            f"{n['edges_boosted']} bike edges boosted ({n['edge_km_boosted']} directed km)"
        )
    print(f"{short_title(name)}: " + "; ".join(parts))

## The change map: Δ jobs reachable

The README's first scenario output. Color shows how many MORE jobs a
hex can reach within 45 minutes by the scenario's mode — grey means no
change (which is most of the city: a single corridor is not a network).
Each map has its own scale: the point is each project's *shape*, not a
cross-project comparison (the gains chart below does that fairly).


In [ ]:
fig, axes = plt.subplots(1, len(ORDER), figsize=(5.4 * len(ORDER), 5.6), squeeze=False)
for ax, name in zip(axes[0], ORDER, strict=True):
    measure, threshold, mode = HEADLINE[name]
    column = f"acc_jobs_{mode}_{threshold}_delta"
    layer = grid.merge(hex_tables[name][["h3", "in_city", column]], on="h3")
    layer = layer[layer["in_city"]]
    gained = layer[layer[column] > 0]
    top = float(np.quantile(gained[column], 0.99)) if len(gained) else 1.0
    layer.plot(ax=ax, color=NEUTRAL, edgecolor=SURFACE, linewidth=0.35, zorder=1)
    if len(gained):
        norm = Normalize(vmin=0.0, vmax=max(top, 1.0))
        gained.plot(
            ax=ax,
            color=[to_hex(SEQ_BLUE(norm(v))) for v in gained[column]],
            edgecolor=SURFACE,
            linewidth=0.35,
            zorder=2,
        )
        bar = fig.colorbar(
            ScalarMappable(norm=norm, cmap=SEQ_BLUE),
            ax=ax,
            orientation="horizontal",
            shrink=0.62,
            pad=0.02,
            aspect=28,
        )
        bar.ax.tick_params(length=0, labelsize=8)
        bar.outline.set_visible(False)
        bar.set_label(f"Δ jobs within {threshold} min by {mode}", fontsize=8.5, color=INK_2)
    for line in scenarios[name].lines:
        geom = to_storage(line.geometry_m)
        ax.plot(*geom.xy, color=INK, linewidth=1.2, alpha=0.75, zorder=5)
    for boost in scenarios[name].boosts:
        geom = to_storage(boost.geometry_m)
        ax.plot(*geom.xy, color=INK, linewidth=1.0, alpha=0.75, zorder=5)
    share = float((layer[column] > 0).mean())
    base_map(ax, layer, f"{short_title(name)}\n{share:.0%} of city hexes gain")
plt.show()

## Who gains: residents whose job access grows by more than X

The README's second output: residents of hexes gaining more than each
cutoff of jobs within 45 minutes — under both population weightings, as
everywhere since Phase 5. The dumbbell shows how much the answer moves
with the population layer alone; the table carries every cutoff.


In [ ]:
cutoffs = sorted(config.scenarios.gain_cutoffs_jobs)
mid = cutoffs[len(cutoffs) // 2]
rows_y = np.arange(len(ORDER))[::-1]
shares = {
    name: {
        w: float(
            gains[
                (gains["scenario"] == name)
                & (gains["cutoff_jobs"] == mid)
                & (gains["weights"] == w)
            ]["share_gaining"].iloc[0]
        )
        * 100
        for w in WEIGHTINGS
    }
    for name in ORDER
}
# A fixed, data-driven x-range BEFORE drawing: with every value at zero a
# bare autoscale collapses the span and tight bounding boxes explode.
right = max(5.0, max(v for d in shares.values() for v in d.values()) * 1.45)

fig, ax = plt.subplots(figsize=(7.6, 0.72 * len(ORDER) + 1.6))
ax.set_xlim(0, right)
for y, name in zip(rows_y, ORDER, strict=True):
    values = shares[name]
    if len(values) == 2:
        ax.hlines(y, *sorted(values.values()), color=AXIS, linewidth=2, zorder=1)
    ax.scatter(
        values[RASTER],
        y,
        s=64,
        color=LIGHT_BLUE,
        edgecolor=SURFACE,
        linewidth=2,
        zorder=3,
        label="raster" if y == rows_y[0] else None,
    )
    if HAS_CALIBRATED:
        ax.scatter(
            values[CALIBRATED],
            y,
            s=64,
            color=BLUE,
            edgecolor=SURFACE,
            linewidth=2,
            zorder=3,
            label="calibrated" if y == rows_y[0] else None,
        )
    low, high = min(values.values()), max(values.values())
    text = f"{low:.1f}-{high:.1f}%" if f"{low:.1f}" != f"{high:.1f}" else f"{high:.1f}%"
    ax.text(high + right * 0.02, y, text, va="center", fontsize=9, color=INK)
ax.set_yticks(rows_y, [short_title(name) for name in ORDER])
ax.set_ylim(-0.6, len(ORDER) - 0.4)
ax.set_xlabel(f"share of residents gaining > {mid:,.0f} jobs within 45 min, %")
share_of_all = mid / report["total_jobs"]
ax.set_title(f"Who gains more than {mid:,.0f} jobs (≈{share_of_all:.0%} of all city jobs)")
ax.legend(loc="lower right")
style(ax, grid="x")
plt.show()

table = gains.pivot_table(
    index=["scenario", "weights"],
    columns="cutoff_jobs",
    values=["residents_gaining", "share_gaining"],
)
with pd.option_context("display.float_format", "{:,.3f}".format):
    print(table.round(3).to_string())

## Does the project reduce inequality? The Gini, before → after

The README's third output. A falling Gini means the new access went
disproportionately to the worse-served; a rising one means the project —
without taking anything from anyone — widened relative gaps. Both can
happen while the *mean* improves, which is exactly why the mean and the
Gini are reported together, and under both weightings: on the full city
the BRT's verdict **flips with the population layer** (raster says more
unequal, calibrated says less), while the metro extension reduces the
Gini under both.


In [ ]:
rows = []
for name in ORDER:
    for weighting in WEIGHTINGS:
        row = summary[
            (summary["scenario"] == name)
            & (summary["weights"] == weighting)
            & (summary["measure"] == HEADLINE[name][0])
        ].iloc[0]
        before, after = float(row["gini_base"]), float(row["gini_scen"])
        if np.isfinite(before) and np.isfinite(after):  # NaN = nobody reaches anything
            rows.append((name, weighting, before, after))
# Fixed x-range before drawing (see the gains chart note).
right = max(0.1, max(max(b, a) for _, _, b, a in rows) * 1.3)

fig, ax = plt.subplots(figsize=(7.6, 0.56 * len(rows) + 1.6))
ax.set_xlim(0, right)
positions = np.arange(len(rows))[::-1]
for y, (_, _, before, after) in zip(positions, rows, strict=True):
    improved = after <= before
    color = BLUE if improved else RED
    ax.hlines(
        y, min(before, after), max(before, after), color=blend(color, 0.45), linewidth=2.4, zorder=1
    )
    ax.scatter(before, y, s=52, color=NEUTRAL, edgecolor=MUTED, linewidth=1.2, zorder=2)
    ax.scatter(after, y, s=64, color=color, edgecolor=SURFACE, linewidth=1.6, zorder=3)
    ax.text(
        max(before, after) + right * 0.02,
        y,
        f"{before:.3f} → {after:.3f} ({after - before:+.4f})",
        va="center",
        fontsize=8.6,
        color=INK,
    )
labels = [f"{short_title(name)} · {weighting}" for name, weighting, _, _ in rows]
ax.set_yticks(positions, labels)
ax.set_ylim(-0.6, len(rows) - 0.4)
ax.set_xlabel("population-weighted Gini of the headline measure (lower = more equal)")
ax.set_title("Inequality before (grey) → after (blue = falls, red = rises)")
handles = [
    plt.Line2D(
        [],
        [],
        marker="o",
        linewidth=0,
        markersize=8,
        markeredgecolor=MUTED,
        markerfacecolor=NEUTRAL,
        label="before",
    ),
    plt.Line2D(
        [],
        [],
        marker="o",
        linewidth=0,
        markersize=8,
        markerfacecolor=BLUE,
        markeredgecolor=SURFACE,
        label="after, Gini falls",
    ),
    plt.Line2D(
        [],
        [],
        marker="o",
        linewidth=0,
        markersize=8,
        markerfacecolor=RED,
        markeredgecolor=SURFACE,
        label="after, Gini rises",
    ),
]
ax.legend(handles=handles, loc="upper left")
style(ax, grid="x")
plt.show()

cols = [
    "scenario",
    "weights",
    "measure",
    "mean_base",
    "mean_scen",
    "median_base",
    "median_scen",
    "gini_base",
    "gini_scen",
    "gini_delta",
]
headline_rows = summary[summary.apply(lambda r: r["measure"] == HEADLINE[r["scenario"]][0], axis=1)]
print(headline_rows[cols].round(4).to_string(index=False))

## The whole distribution: Lorenz before and after

The Gini compresses the change into one number; the Lorenz curve shows
where on the distribution it happened. Drawn for the scenario with the
largest mean gain, raster weighting: if the "after" curve pulls toward
the diagonal near the left, the worst-served gained share; if it only
moves in the middle, the project mostly served the already-served.


In [ ]:
headline_rows = summary[summary.apply(lambda r: r["measure"] == HEADLINE[r["scenario"]][0], axis=1)]
raster_rows = headline_rows[headline_rows["weights"] == RASTER]
focus = raster_rows.loc[(raster_rows["mean_scen"] - raster_rows["mean_base"]).idxmax(), "scenario"]
measure, threshold, mode = HEADLINE[focus]

features = pd.read_parquet(P / f"features_r{A}.parquet", columns=["h3", "pop_residents"])
table = hex_tables[focus].merge(features, on="h3")
table["pop"] = table["pop_residents"].fillna(0.0)
domain = table[table["in_city"] & (table["pop"] >= config.equity.min_population)]
weights = domain["pop"].to_numpy()

fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.plot([0, 1], [0, 1], color=AXIS, linewidth=1)
for suffix, color, label in (
    (f"{measure}_base", MUTED, "baseline"),
    (f"{measure}_scen", BLUE, "with the project"),
):
    values = domain[suffix].to_numpy()
    pop_share, access_share = lorenz_curve(values, weights)
    if np.isnan(access_share).all():
        continue
    ax.plot(
        pop_share,
        access_share,
        color=color,
        linewidth=2,
        solid_capstyle="round",
        label=f"{label} (Gini {gini(values, weights):.3f})",
    )
ax.set_aspect("equal")
ax.set_xticks([0, 0.5, 1], ["0", "50%", "100%"])
ax.set_yticks([0, 0.5, 1], ["0", "50%", "100%"])
ax.set_xlabel("share of residents, worst served first")
ax.set_ylabel("share of all job access")
ax.set_title(f"{short_title(focus)} — {measure_label(measure)}, raster weighting")
ax.legend(loc="upper left")
style(ax, grid="both")
plt.show()

## Checks

Every claim above, re-derived from the raw tables: baselines must equal
the Phase 4 access table, deltas must be non-negative and internally
consistent, the gains must reproduce from per-hex deltas and residents,
the scenario's "before" Gini must equal the equity report's Gini, the
scenario matrices can only be larger than the baseline's, and the report
must state its caveats.


In [ ]:
failures = []


def check(name: str, condition: bool) -> None:
    print(("PASS " if condition else "FAIL ") + name)
    if not condition:
        failures.append(name)


features = pd.read_parquet(P / f"features_r{A}.parquet", columns=["h3", "pop_residents"])
population = (
    features.set_index("h3")["pop_residents"].reindex(pd.Index(access["h3"])).fillna(0.0).to_numpy()
)
inhabited = access["in_city"].to_numpy(dtype=bool) & (population >= config.equity.min_population)
thresholds = config.travel.thresholds_min

# 1. Baseline columns are the Phase 4 access table, bit for bit.
ok_base, ok_sign, ok_sum, some_gain = True, True, True, False
for name, table in hex_tables.items():
    for mode in report["scenarios"][name]["modes"]:
        for column in affected_columns(mode, thresholds):
            ok_base &= bool(
                np.array_equal(table[f"{column}_base"].to_numpy(), access[column].to_numpy())
            )
            delta = table[f"{column}_delta"].to_numpy()
            ok_sign &= bool((delta >= 0).all())
            ok_sum &= bool(
                np.allclose(delta, table[f"{column}_scen"].to_numpy() - access[column].to_numpy())
            )
            some_gain |= bool((delta > 0).any())
check("baseline columns equal the stored access table", ok_base)
check("no delta is negative (adding arcs can only help)", ok_sign)
check("delta = scenario - baseline everywhere", ok_sum)
check("at least one scenario improves at least one hex", some_gain)

# 2. Gains reproduce from per-hex deltas and residents. Calibrated
# weights are re-derived with the library (the report's factors are
# rounded for display and drift by hundreds of residents at city scale).
from almaty_access.equity.weights import (  # noqa: E402
    apply_factors,
    calibrated_weights,
    district_resident_shares,
)

weight_arrays = {RASTER: population}
if HAS_CALIBRATED:
    calibration = district_resident_shares(config)
    _, factors = calibrated_weights(
        pd.Series(population[inhabited]),
        access.loc[inhabited, "district"],
        calibration.shares,
    )
    weight_arrays[CALIBRATED] = apply_factors(
        pd.Series(population), access["district"].where(access["in_city"]), factors
    )
ok_gains = True
for name in ORDER:
    measure, threshold, mode = HEADLINE[name]
    delta = hex_tables[name][f"acc_jobs_{mode}_{threshold}_delta"].to_numpy()
    for weighting in WEIGHTINGS:
        w = weight_arrays[weighting]
        mine = gains[(gains["scenario"] == name) & (gains["weights"] == weighting)]
        for row in mine.itertuples():
            expected = float(w[inhabited & (delta > row.cutoff_jobs)].sum())
            tolerance = max(1e-6 * max(expected, 1), 0.5)
            ok_gains &= abs(row.residents_gaining - expected) <= tolerance
check("gains reproduce from per-hex deltas and residents", ok_gains)

# 3. Gains can only shrink as the cutoff grows.
monotone = True
for (_, _), group in gains.groupby(["scenario", "weights"]):
    ordered = group.sort_values("cutoff_jobs")["residents_gaining"].to_numpy()
    monotone &= bool((np.diff(ordered) <= 1e-9).all())
check("a higher bar never admits more residents", monotone)

# 4. The scenario's Gini-before equals the equity report's Gini.
equity_summary = pd.read_parquet(P / EQUITY_SUMMARY_FILE)
ok_gini = True
for name in ORDER:
    measure = HEADLINE[name][0]
    for weighting in WEIGHTINGS:
        ours = summary[
            (summary["scenario"] == name)
            & (summary["weights"] == weighting)
            & (summary["measure"] == measure)
        ]["gini_base"]
        theirs = equity_summary[
            (equity_summary["res"] == A)
            & (equity_summary["weights"] == weighting)
            & (equity_summary["measure"] == measure)
        ]["gini"]
        both_nan = ours.isna().all() and theirs.isna().all()
        ok_gini &= both_nan or bool(np.isclose(ours.iloc[0], theirs.iloc[0], rtol=1e-9))
check("scenario Gini-before equals the Phase 5 equity Gini", ok_gini)

# 5. Adding arcs can only grow the within-cutoff pair set.
tt_stats = json.loads((P / "tt_stats.json").read_text("utf-8"))
ok_pairs = True
for name in ORDER:
    for mode, stats in report["scenarios"][name]["matrix"].items():
        baseline_pairs = tt_stats[f"tt_{mode}_r{A}.parquet"]["pairs"]
        ok_pairs &= stats["pairs"] >= baseline_pairs
check("scenario matrices hold at least the baseline's pairs", ok_pairs)


# 6. Summary table is complete and the report states its caveats.
def _relative(mode: str) -> list[str]:
    columns = affected_columns(mode, thresholds)
    return [c for c in columns if c.startswith("accpct_") or c.endswith("_grav")]


expected_rows = sum(
    len(WEIGHTINGS) * len(_relative(m))
    for name in ORDER
    for m in report["scenarios"][name]["modes"]
)
check("summary covers every scenario x weighting x measure", len(summary) == expected_rows)
json.dumps(report, allow_nan=False)
any_transit = any("transit" in report["scenarios"][name]["modes"] for name in ORDER)
check(
    "report states the lower-bound baseline caveat",
    (not any_transit) or any("lower-bound" in c for c in report["caveats"]),
)
check(
    "every scenario carries the fixed-jobs caveat",
    all(
        any("job surface" in c for c in entry["caveats"]) for entry in report["scenarios"].values()
    ),
)

print()
if failures:
    raise AssertionError(f"verdict failed: {failures}")
print("All checks passed.")
for name in ORDER:
    print("RESULT:", report["scenarios"][name]["headline"])

## What to remember

- A **scenario is data** (GeoJSON): alignment + assumed speed/headway.
  Redrawing one requires no code. Provenance and sources live in the
  file and are echoed into the report.
- The engine's two rails: **nothing drawn reproduces Phase 4 exactly**,
  and **deltas are provably non-negative** — both enforced by tests,
  the second also at runtime.
- An extension **rides the same train** (no phantom transfer wait);
  a new line pays its own expected wait at boarding; bike corridors
  raise the flat speed only on edges that mostly lie inside the buffer.
- Reach and equity are different axes: on the full city the BRT helps
  ~17–19% of residents but its Gini verdict flips between population
  weightings; the metro extension helps ~1% yet lowers the Gini under
  both. Quote both, always.
- Everything is measured against the **lower-bound transit baseline**
  (D-044): gains are overstated where the baseline misses real routes.

Details and alternatives: `docs/LEARNING_GUIDE.md` (Phase 6 chapter) and
`DECISIONS.md` D-065…D-068.
